# EDA Insights

### DOMAIN: ECOMMERCE | INDIAN HERBAL SKIN/HAIR CARE PRODUCTS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('master_orders.csv', parse_dates=['Created at'])
orders = pd.read_csv('orders.csv')
orders_raw = pd.read_csv('orders_raw.csv')
rfm = pd.read_csv('rfm.csv')

In [ ]:
df.head()

### 1. Revenue Health Audit

1.1. Total Revenue Breakdown

In [ ]:
revenue_breakdown = df.groupby('Financial Status')['Net Revenue'].sum().sort_values(ascending=False).round(2)
print("\nRevenue by Financial Status:")
print(revenue_breakdown)

1.2. Net Revenue vs Gross

In [ ]:
df['Gross Revenue'] = orders['Total Price']
orders['Gross Revenue'] = df['Gross Revenue']

In [ ]:
leakage = df['Gross Revenue'].sum() - df['Net Revenue'].sum()

1.3. AOV & Key Metrics

In [ ]:
metrics = {
    'AOV': df['Net Revenue'].mean(),
    'Orders' : len(df),
    'Customers' : df['Customer ID'].nunique(),
    'Repeat Rate': df[df['Customer ID']>0]['Customer ID'].value_counts().gt(1).mean(),
    'Refund Rate' : len(orders_raw[orders_raw['Financial Status'] == 'refunded']) / len(orders_raw)}

metric_df = pd.DataFrame(list(metrics.items()), columns=['Metric', 'Value'])  
metric_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].pie(revenue_breakdown.values, labels=revenue_breakdown.index, autopct='%1.2f%%', startangle=90)
axes[0].set_title('Revenue by Financial Status')

axes[1].bar(revenue_breakdown.index,revenue_breakdown.values)
axes[1].set_title('Net Revenue by Financial Status')
axes[1].set_ylabel('Net Revenue')
axes[1].tick_params(axis='x', rotation=0)

sns.barplot(data=metric_df, x='Metric', y='Value', ax=axes[2])
axes[2].set_title('Core Ecommerce KPIs')
plt.tight_layout()
plt.savefig('revenue_audit.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
total_revenue = orders['Net Revenue'].sum()
total_discount = orders['Total Discounts'].sum()
revenue_leakage_pct = (total_discount / (total_revenue + total_discount)) * 100


gross_revenue = total_revenue + total_discount
net_revenue = total_revenue

labels = ['Gross Revenue', 'Discounts', 'Net Revenue']
values = [gross_revenue, -total_discount, net_revenue]

cumulative = [0, gross_revenue, gross_revenue - total_discount]

plt.figure()
plt.bar(labels, values)
plt.axhline(0)
plt.title('Revenue Leakage Breakdown (Waterfall)')
plt.ylabel('Revenue')

plt.show()


### 2. Time Series Analysis

In [ ]:
rfm

In [ ]:
orders

1. Monthly Revenue trend

In [ ]:
fig = px.line(orders, x='Order Month', y='Net Revenue', 
              title='Monthly Revenue Trend & Growth Rate')
fig.add_hline(y=orders['Net Revenue'].mean(), line_dash="dash", 
              annotation_text="Avg Monthly Revenue")
fig.write_html('monthly_trend.html')
fig.show()

2. Seasonality heatmap

In [ ]:
heatmap_data = orders.groupby(['Order Year', 'Order week'])['Net Revenue'].sum().reset_index()
heatmap_matrix = heatmap_data.pivot(
    index='Order Year',
    columns='Order week',
    values='Net Revenue'
)

In [ ]:
plt.figure(figsize=(14, 4))
sns.heatmap(heatmap_matrix, annot=False, cmap='YlOrRd')
plt.title('Revenue Heatmap (Seasonality)')
plt.xlabel('Week Number')
plt.ylabel('Year')
plt.tight_layout()
plt.savefig('seasonality_heatmap.png', dpi=300)
plt.show()

3. Customer Segmentation

In [ ]:
snapshot_date = df['Created at'].max()
rfm = df.groupby('Customer ID').agg({
    'Created at': lambda x: (snapshot_date - x.max()).days,  # Recency
    'Order ID': 'count',  # Frequency
    'Net Revenue': 'sum'  # Monetary
}).round(0).rename(columns={'Created at': 'Recency', 'Order ID': 'Frequency', 'Net Revenue': 'Monetary'})


rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'], q=5, labels=False, duplicates='drop') + 1  
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)


In [ ]:
segment_rev = rfm.groupby('RFM_Score')['Monetary'].sum().sort_values(ascending=False)
print("Top 5 RFM Segments by Revenue Contribution:")
print(segment_rev.head())

In [ ]:
fig = make_subplots(rows=2, cols=2, 
                    subplot_titles=('RFM Score Distribution', 'Revenue by RFM Segment', 
                                  'Recency vs Frequency', 'Monetary Distribution'))
fig.add_trace(go.Histogram(x=rfm['RFM_Score'], name='RFM Dist'), row=1, col=1)
fig.add_trace(go.Bar(x=segment_rev.index[:10], y=segment_rev.values[:10], name='Top Segments'), row=1, col=2)
fig.add_trace(go.Scatter(x=rfm['Recency'], y=rfm['Frequency'], mode='markers', name='RFM Scatter'), row=2, col=1)
fig.add_trace(go.Histogram(y=rfm['Monetary'], name='Monetary'), row=2, col=2)
fig.update_layout(height=600, title_text="RFM Customer Segmentation")
fig.write_html('rfm_preview.html')
fig.show()

4. Product Profitability

In [ ]:
print("Top 5 Most Profitable SKUs:")
sku_summary = orders.groupby('Lineitem name').agg({
    'Gross Revenue' : 'sum',
    'Lineitem Gross Profit': 'sum',
    'Gross Margin %': 'mean'}).reset_index()
print(sku_summary.sort_values('Lineitem Gross Profit', ascending=False).head(5))

In [ ]:
top_skus = sku_summary.sort_values('Lineitem Gross Profit', ascending=False).head(10).copy()
top_skus['Lineitem name short'] = top_skus['Lineitem name'].str.slice(0, 30)

sku_summary['cumulative_profit_pct'] = sku_summary['Lineitem Gross Profit'].rank(ascending=False) / len(sku_summary)
pareto_skus = sku_summary[sku_summary['cumulative_profit_pct'] <= 0.8]

In [ ]:
top_skus = (
    sku_summary
    .sort_values('Lineitem Gross Profit', ascending=False)
    .head(10)
    .copy()
)

top_skus['Lineitem name short'] = top_skus['Lineitem name'].str.slice(0, 30)
sku_sorted = sku_summary.sort_values(
    'Lineitem Gross Profit', ascending=False
).reset_index(drop=True)

sku_sorted['cumulative_profit'] = sku_sorted['Lineitem Gross Profit'].cumsum()
sku_sorted['cumulative_profit_pct'] = (
    sku_sorted['cumulative_profit'] /
    sku_sorted['Lineitem Gross Profit'].sum()
) * 100

sku_sorted['sku_pct'] = (
    (sku_sorted.index + 1) / len(sku_sorted)
) * 100
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes[0, 0].scatter(
    sku_summary['Gross Revenue'],
    sku_summary['Lineitem Gross Profit'],
    alpha=0.7
)

axes[0, 0].axhline(0)
axes[0, 0].set_xlabel('Revenue')
axes[0, 0].set_ylabel('Gross Profit')
axes[0, 0].set_title('Revenue vs Profit by SKU')
axes[0, 1].boxplot(sku_summary['Gross Margin %'])
axes[0, 1].set_ylabel('Gross Margin %')
axes[0, 1].set_title('SKU Profit Margin Distribution')
axes[1, 0].barh(
    top_skus['Lineitem name short'],
    top_skus['Lineitem Gross Profit']
)

axes[1, 0].invert_yaxis()
axes[1, 0].set_xlabel('Gross Profit')
axes[1, 0].set_ylabel('SKU')
axes[1, 0].set_title('Top Profitable SKUs')
axes[1, 1].plot(
    sku_sorted['sku_pct'],
    sku_sorted['cumulative_profit_pct'],
    marker='o'
)

axes[1, 1].axhline(80)
axes[1, 1].axvline(20)

axes[1, 1].set_xlabel('% of SKUs')
axes[1, 1].set_ylabel('Cumulative Profit %')
axes[1, 1].set_title('SKU Profit Concentration (Pareto)')
plt.tight_layout()
plt.savefig('sku_profitability.png', dpi=300, bbox_inches='tight')
plt.show()
